In [1]:

from pyspark.sql import SparkSession


In [2]:
# Create the SparkSession instance required for this notebook
spark = SparkSession.builder \
    .appName("Lab02_DataFrame_Transformations") \
    .getOrCreate()

# Set log level to minimize noise
spark.sparkContext.setLogLevel("ERROR")

print("Spark Session successfully initialized for Notebook 2!")

Spark Session successfully initialized for Notebook 2!


## Create Data

In [3]:

employees = spark.createDataFrame([
    (1, "Ali", 3000),
    (2, "Sara", 7000),
    (3, "Omar", None),
    (4, "Mona", 10000)
], ["id", "name", "salary"])

departments = spark.createDataFrame([
    (1, "HR"),
    (2, "IT"),
    (2, "IT_DUPLICATE"),
    (3, "Finance"),
], ["id", "department"])

employees.show()
departments.show()


+---+----+------+
| id|name|salary|
+---+----+------+
|  1| Ali|  3000|
|  2|Sara|  7000|
|  3|Omar|  null|
|  4|Mona| 10000|
+---+----+------+

+---+------------+
| id|  department|
+---+------------+
|  1|          HR|
|  2|          IT|
|  2|IT_DUPLICATE|
|  3|     Finance|
+---+------------+




## Join the two dataframe


In [4]:

# Perform an inner join between employees and departments based on the 'id' column
joined_df = employees.join(departments, on="id", how="inner")

# Display the resulting DataFrame
joined_df.show()

+---+----+------+------------+
| id|name|salary|  department|
+---+----+------+------------+
|  1| Ali|  3000|          HR|
|  3|Omar|  null|     Finance|
|  2|Sara|  7000|          IT|
|  2|Sara|  7000|IT_DUPLICATE|
+---+----+------+------------+




### Debug Task:
- Why does id=2 appear multiple times?
- What kind of data issue caused this?


## Fix Join Issue

In [5]:

# 1. Fix the data issue by dropping duplicates from the departments DataFrame based on 'id'
clean_departments = departments.dropDuplicates(["id"])

# 2. Re-perform the inner join using the deduplicated departments DataFrame
fixed_joined_df = employees.join(clean_departments, on="id", how="inner")

# Display the fixed result
print("Fixed Join Result (No Duplicates):")
fixed_joined_df.show()

Fixed Join Result (No Duplicates):
+---+----+------+----------+
| id|name|salary|department|
+---+----+------+----------+
|  1| Ali|  3000|        HR|
|  3|Omar|  null|   Finance|
|  2|Sara|  7000|        IT|
+---+----+------+----------+



## categorize the employee salary with new column using Case When

In [6]:

from pyspark.sql.functions import col, when

# Categorize the employee salary into groups using when/otherwise (Case When)
categorized_df = fixed_joined_df.withColumn(
    "salary_category",
    when(col("salary") < 5000, "Low")
    .when((col("salary") >= 5000) & (col("salary") <= 9000), "Medium")
    .when(col("salary") > 9000, "High")
    .otherwise("Unknown")  # Handles null or missing values safely
)

# Display the final results showing the new column
print("Final Categorized DataFrame:")
categorized_df.show()

Final Categorized DataFrame:
+---+----+------+----------+---------------+
| id|name|salary|department|salary_category|
+---+----+------+----------+---------------+
|  1| Ali|  3000|        HR|            Low|
|  3|Omar|  null|   Finance|        Unknown|
|  2|Sara|  7000|        IT|         Medium|
+---+----+------+----------+---------------+

